# Assignment 2 — Add Memory to Your Agent

Goal: extend the Assignment 1 agent with memory so it recalls past
interactions within a session. This notebook re-implements the same
search/calculator agent and adds:

1. A simple key-value / list-based memory (`SimpleMemory`)
2. An equivalent built using LangChain's `ConversationBufferMemory`

Both versions are demonstrated with a multi-turn conversation where the
second question only makes sense in light of the first.


## 0. Setup

In [1]:
!pip install -q transformers accelerate ddgs langchain-core


In [2]:
import re
import ast
import operator as op

from transformers import pipeline


## 1. Model, tools, and the ReAct loop (same as Assignment 1)

In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # <- lighter alternative for CPU/local testing

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.0,
)


def llm(prompt: str) -> str:
    out = generator(prompt, max_new_tokens=300, return_full_text=False)
    return out[0]["generated_text"].strip()


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
from ddgs import DDGS


def search_tool(query: str, max_results: int = 3) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        return "\n".join(f"- {r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search error: {e}"


_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.UAdd: op.pos,
    ast.FloorDiv: op.floordiv,
    # Note: '%' is reserved for percent notation (see _preprocess_percent below),
    # not modulo - that avoids ambiguity with expressions like '10%*69081996'.
}

PERCENT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%")


def _preprocess_percent(expression: str) -> str:
    """Turn 'N%' into '(N/100)' so the model can write percentages naturally,
    e.g. '10%*69081996' -> '(10/100)*69081996'."""
    return PERCENT_RE.sub(r"(\1/100)", expression)


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression node: {ast.dump(node)}")


def calculator_tool(expression: str) -> str:
    try:
        expression = _preprocess_percent(expression)
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


TOOLS = {"search": search_tool, "calculator": calculator_tool}
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)


## 2. Simple key-value memory

`SimpleMemory` stores each (question, answer) turn and renders the recent
history as plain text that gets prepended to the agent's prompt, giving the
model contextual continuity across turns.


In [5]:
class SimpleMemory:
    def __init__(self):
        self.history: list[dict] = []   # [{"question": ..., "answer": ...}, ...]

    def add(self, question: str, answer: str) -> None:
        self.history.append({"question": question, "answer": answer})

    def as_context(self, max_turns: int = 5) -> str:
        recent = self.history[-max_turns:]
        if not recent:
            return "(no previous conversation)"
        lines = []
        for turn in recent:
            lines.append(f"User: {turn['question']}")
            lines.append(f"Agent: {turn['answer']}")
        return "\n".join(lines)

    def clear(self) -> None:
        self.history = []


In [6]:
SYSTEM_PROMPT = """You are a helpful AI agent that solves problems by reasoning step by step
using the ReAct (Reasoning + Acting) pattern. You remember the conversation
so far and should use it to resolve references like "that", "it", or "my
previous question".

Tools available:
- search[query]: searches the web for current or factual information.
- calculator[expression]: evaluates a math expression.

Respond using EXACTLY this format, one step at a time:

Thought: <your reasoning about what to do next>
Action: <tool_name>[<tool_input>]

Wait for the Observation before continuing. When you have enough information:

Thought: I now know the final answer.
Final Answer: <the answer to the original question>

Conversation so far:
{history}

Begin!

Question: {question}
"""


def run_agent(question: str, memory: SimpleMemory, max_steps: int = 5, verbose: bool = True) -> str:
    transcript = SYSTEM_PROMPT.format(history=memory.as_context(), question=question)

    for step in range(max_steps):
        response = llm(transcript)
        response = response.split("Observation:")[0].strip()
        if verbose:
            print(f"--- step {step + 1} ---")
            print(response)

        transcript += "\n" + response

        if "Final Answer:" in response:
            answer = response.split("Final Answer:")[-1].strip()
            memory.add(question, answer)
            return answer

        match = ACTION_RE.search(response)
        if not match:
            answer = "Agent stopped: no valid Action or Final Answer was produced."
            memory.add(question, answer)
            return answer

        tool_name, tool_input = match.group(1).strip(), match.group(2).strip()
        tool_fn = TOOLS.get(tool_name)
        observation = tool_fn(tool_input) if tool_fn else f"Unknown tool '{tool_name}'."

        obs_line = f"\nObservation: {observation}\n"
        if verbose:
            print(obs_line)
        transcript += obs_line

    answer = "Agent stopped: reached max_steps without a Final Answer."
    memory.add(question, answer)
    return answer


### Demo: multi-turn recall

In [7]:
memory = SimpleMemory()

print("Turn 1")
a1 = run_agent("My name is Priya and I'm researching the population of France.", memory)
print("\nANSWER 1:", a1)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 1


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- step 1 ---
Assistant: Thought: Understand the context of the conversation.
Assistant: Thought: Priya is researching the population of France.
Assistant: Action: search[Population of France]


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Observation: - Population of France: The demography of France is monitored by the Institut national d'études démographiques (INED) and the Institut national de la statistique et des études économiques (INSEE). As of 1 January 2026, in Metropolitan France 66,792,845 people lived, while 2,289,151 lived in overseas France, for a total of 69,081,996 inhabitants in the French Republic. In the 2010s and until 2017, the population of France grew by 1 million people every three years – an average annual increase of 340,000 people, or +0.6%.France was historically Europe's most populous country. During the Middle Ages, more than one-quarter of Europe's total population was French; by the seventeenth century, this had decreased slightly to one-fifth. By the beginning of the nineteenth century, other European countries, such as Germany and Russia, had caught up with France and overtaken it in number of people. The country's population sharply increased with the baby boom following World War II, 

In [8]:
print("Turn 2")
a2 = run_agent("What's my name, and what would 10% of that population figure be?", memory)
print("\nANSWER 2:", a2)


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 2


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- step 1 ---
Thought: The user has introduced themselves as Priya. The population figure is 69,081,996. To find 10% of the population, I need to calculate 10% of 69,081,996.

Action: calculator[10%*69081996]

Observation: 6908199.600000001

--- step 2 ---
Thought: I now know the final answer.
Final Answer: The name is Priya and 10% of the population of France is approximately 6,908,199.6 people.

ANSWER 2: The name is Priya and 10% of the population of France is approximately 6,908,199.6 people.


In [9]:
print("Memory contents so far:")
for i, turn in enumerate(memory.history, 1):
    print(f"{i}. Q: {turn['question']}\n   A: {turn['answer']}\n")


Memory contents so far:
1. Q: My name is Priya and I'm researching the population of France.
   A: The population of France was 69,081,996 as of January 1, 2026.

2. Q: What's my name, and what would 10% of that population figure be?
   A: The name is Priya and 10% of the population of France is approximately 6,908,199.6 people.



## 3. Alternative: LangChain's message-history memory

The same idea, expressed with LangChain's own memory primitive instead of a
hand-rolled class.

Note: `langchain.memory.ConversationBufferMemory` (the class historically
used for this) has been **deprecated and removed from the main `langchain`
package** in current releases - LangChain's own migration guide now points
people toward `langchain_core`'s message-history primitives or LangGraph
persistence instead. We use `InMemoryChatMessageHistory` here, which is the
lightweight, non-deprecated equivalent: it stores typed `HumanMessage` /
`AIMessage` objects and is part of `langchain-core`, so it doesn't depend on
the legacy `langchain.memory` module at all.

(If you specifically need the old `ConversationBufferMemory` class for
compatibility with older code, it still exists in the separate
`langchain_classic` package: `pip install langchain_classic` then
`from langchain_classic.memory import ConversationBufferMemory`.)


In [10]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

lc_memory = InMemoryChatMessageHistory()

# Record turn 1 (reusing the answer we already generated with SimpleMemory above)
lc_memory.add_user_message("My name is Priya and I'm researching the population of France.")
lc_memory.add_ai_message(a1)


def render_history(history: InMemoryChatMessageHistory) -> str:
    if not history.messages:
        return "(no previous conversation)"
    lines = []
    for msg in history.messages:
        role = "User" if isinstance(msg, HumanMessage) else "Agent"
        lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)


print(render_history(lc_memory))


User: My name is Priya and I'm researching the population of France.
Agent: The population of France was 69,081,996 as of January 1, 2026.


In [11]:
def run_agent_lc_memory(question: str, history: InMemoryChatMessageHistory,
                         max_steps: int = 5, verbose: bool = True) -> str:
    history_text = render_history(history)
    transcript = SYSTEM_PROMPT.format(history=history_text, question=question)

    for step in range(max_steps):
        response = llm(transcript)
        response = response.split("Observation:")[0].strip()
        if verbose:
            print(f"--- step {step + 1} ---")
            print(response)
        transcript += "\n" + response

        if "Final Answer:" in response:
            answer = response.split("Final Answer:")[-1].strip()
            history.add_user_message(question)
            history.add_ai_message(answer)
            return answer

        match = ACTION_RE.search(response)
        if not match:
            answer = "Agent stopped: no valid Action or Final Answer was produced."
            history.add_user_message(question)
            history.add_ai_message(answer)
            return answer

        tool_name, tool_input = match.group(1).strip(), match.group(2).strip()
        tool_fn = TOOLS.get(tool_name)
        observation = tool_fn(tool_input) if tool_fn else f"Unknown tool '{tool_name}'."
        obs_line = f"\nObservation: {observation}\n"
        if verbose:
            print(obs_line)
        transcript += obs_line

    answer = "Agent stopped: reached max_steps without a Final Answer."
    history.add_user_message(question)
    history.add_ai_message(answer)
    return answer


answer_lc = run_agent_lc_memory(
    "What's my name, and what would 10% of that population figure be?", lc_memory
)
print("\nANSWER (LangChain memory):", answer_lc)


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- step 1 ---
Thought: The user has introduced themselves as Priya. The population figure is 69,081,996. To find 10% of the population, I need to calculate 10% of 69,081,996.

Action: calculator[10%*69081996]

Observation: 6908199.600000001

--- step 2 ---
Thought: I now know the final answer.
Final Answer: The name is Priya and 10% of the population of France is approximately 6,908,199.6 people.

ANSWER (LangChain memory): The name is Priya and 10% of the population of France is approximately 6,908,199.6 people.


## Notes / extensions

- `SimpleMemory.as_context(max_turns=...)` caps how much history is
  injected into the prompt - useful once conversations get long and you
  start hitting the model's context window.
- For longer-running assistants, consider `ConversationSummaryMemory`
  (LangChain) which periodically compresses old turns into a summary
  instead of keeping the full transcript.
